# FA15_PERTURB — F-A1.5, project verification (not part of the report; board R204)

**Plan:** `docs/FUTURE_RESEARCH_PLAN.md` §12 15/11 12:40; `results/fa15/DESIGN_D35F.md`. **Does:** diagnostic T2 (src/fa15_perturb.py): the 70 V9-X6 files twice, as is and x (1 + 2^-40); ASR exactly as V9-X6 (asrpy 0.0.8, cutoff 20, per-file
calibration, one process per file) on the pipeline-filtered signal, then the record's statistics and features on the
cleaned signal; pre-gates —;
gates G-conv (reported; STOP > 1e-2), G-lab, G-asr (differences explained by V9-X6's earlier chunkings), G-done,
G-worker-sha.
**Cases:** CHB-MIT —; external —.
**Accelerator:** None (CPU). **Internet:** ON. **Inputs:** `bonesclarke26/combined-siena-scalp-and-chb-mit-eeg-datasets`, `thesis-r2-chbmit-features` (nhn2mm),
`thesis-r2-external-features` (norncreades).
**Runtime:** ≈ 2 h (computed: 88.5 h of EEG, two ASR passes, 4 processes, FA15_DATA_A's measured 146 s per hour of EEG per process). Code: `github.com/Nhanvas/Thesis` at `9682471`; `src/fa15_asr.py` sha256 `f9a548e45cfed9a6` (checked).
A gate failure is recorded; the END cell archives the results, then raises `STOPPED: ...`.
Do not edit; *Save Version → Save & Run All (Commit)*.

In [ ]:
# SETUP: packages + repository at the pinned commit (read only)
import os, sys, subprocess, time, json, hashlib, shutil
T_NB = time.time()
os.environ["FA15_T0"] = os.environ["R2_T0"] = str(T_NB)
NOTEBOOK, REPO_COMMIT, TAG = "FA15_PERTURB", "9682471a483ed33282a700f2c4ae98ff5fd6e3c9", "perturb"
TEST = bool(os.environ.get("FA15_TEST_REPO"))                    # synthetic test in the build container only
WORK = os.environ.get("FA15_TEST_WORK", "/kaggle/working")
INPUT = os.environ.get("FA15_TEST_INPUT", "/kaggle/input")
SCRATCH = os.environ.get("FA15_TEST_SCRATCH", "/tmp/fa15")
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[v] = "1"
if TEST:
    REPO = os.environ["FA15_TEST_REPO"]
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "asrpy==0.0.8", "mne==1.12.1", "torch_geometric==2.7.0", "ruptures==1.1.10", "timescoring==0.0.7"])
    REPO = "/tmp/Thesis"
    if not os.path.exists(REPO + "/src/fa15_asr.py"):
        r = subprocess.run(f"git init -q {REPO} && cd {REPO} && git remote add origin https://github.com/Nhanvas/Thesis.git "
                           f"&& git fetch -q --depth 1 origin {REPO_COMMIT} && git checkout -q FETCH_HEAD", shell=True)
        if r.returncode != 0:
            subprocess.check_call(f"rm -rf {REPO} && git clone -q https://github.com/Nhanvas/Thesis.git {REPO} "
                                  f"&& cd {REPO} && git checkout -q {REPO_COMMIT}", shell=True)
    head = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"]).decode().strip()
    assert head == REPO_COMMIT, f"wrong commit {head}"
sha = hashlib.sha256(open(REPO + "/src/fa15_asr.py", "rb").read()).hexdigest()[:16]
assert TEST or sha == "f9a548e45cfed9a6", f"fa15_asr.py sha {sha} != f9a548e45cfed9a6"
if True:
    shp = hashlib.sha256(open(REPO + "/src/fa15_perturb.py", "rb").read()).hexdigest()[:16]
    assert TEST or shp == "a6d2e54bc1c0afbe", f"fa15_perturb.py sha {shp} != a6d2e54bc1c0afbe"
os.environ.update(THESIS_REPO=REPO, FA15_WORK=WORK, FA15_SCRATCH=SCRATCH, R2_WORK=WORK, R2_INPUT=INPUT,
                  R2_SCRATCH=SCRATCH + "/r2")
sys.path.insert(0, REPO + "/src")
print("repository", REPO, "CPU cores", os.cpu_count(), f"({time.time()-T_NB:.0f}s)")

In [ ]:
# RUN: diagnostic T2 -- ASR on the 70 V9-X6 files twice (as is; x (1 + 2^-40)), Measure 1, comparisons, gates
import fa15_perturb as FP
TAG = "perturb"
FAILED = FP.run(notebook=NOTEBOOK, commit=REPO_COMMIT)
print("failed gates:", FAILED or "none")

In [ ]:
# END: archive the results (small files), measured runtime; datasets stay in WORK for "Output -> New Dataset"
OUT = os.path.join(WORK, "fa15", TAG)
for d in ("specs",):
    shutil.rmtree(os.path.join(SCRATCH, d), ignore_errors=True)
ZIP = shutil.make_archive(os.path.join(WORK, f"fa15_{TAG}_results"), "zip", root_dir=os.path.join(WORK, "fa15"), base_dir=TAG)
print("results archive:", ZIP, f"{os.path.getsize(ZIP) / 1e6:.1f} MB")
for dname in sorted(os.listdir(WORK)):
    if dname.startswith("fa15-asr-"):
        fs = os.listdir(os.path.join(WORK, dname))
        print("dataset folder", dname, len(fs), "files", f"{sum(os.path.getsize(os.path.join(WORK, dname, f)) for f in fs) / 1e9:.2f} GB")
print(open(os.path.join(OUT, "SUMMARY.md")).read())
print("MEASURED notebook runtime (h):", round((time.time() - T_NB) / 3600, 3))
if FAILED:
    raise RuntimeError(f"STOPPED: failed gates {FAILED} (results archived)")